# 🛠️ 03 — Skill Analysis: Was wird im Data-Markt gefragt?

**Ziel:** Verstehen, welche Skills in deutschen Data-Stellen gefragt sind und wie sie zusammenhängen.

## 🤔 Forschungsfragen

1. Welche Skills sind die **Top 10** im Markt?
2. Welche Skills **kommen oft zusammen vor**? (Co-Occurrence)
3. Wie unterscheiden sich die Skill-Anforderungen **pro Rolle**?
4. Gibt es **Junior-spezifische Skill-Profile**?
5. Welche Skills sind in **Remote-Stellen** stärker gefragt?

## 📊 Methodik

Wir nutzen die bereits **bereinigten und mit Skills angereicherten Daten** aus der Pipeline. Skills wurden mit Wortgrenzen-Regex aus Job-Beschreibungen extrahiert und über Synonym-Mapping normalisiert (`powerbi` → `power bi`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 20)

In [ ]:
# Daten laden
jobs = pd.read_csv(ROOT / "data" / "processed" / "jobs_cleaned.csv")
skills = pd.read_csv(ROOT / "data" / "processed" / "job_skills.csv")

# Nur klassifizierte Data-Jobs für die Analyse (Other ausschließen)
jobs = jobs[jobs["role_group"] != "Other"].copy()
skills = skills[skills["job_id"].isin(jobs["job_id"])].copy()

print(f"📋 Jobs:    {len(jobs):>5,}")
print(f"🛠️ Skills:  {skills['skill'].nunique():>5,} eindeutige (von {len(skills):,} Erwähnungen)")

## 1️⃣ Top-Skills: Was wird am häufigsten gesucht?

In [ ]:
# Top 15 Skills nach absoluter Häufigkeit
top_skills = skills["skill"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    x=top_skills.values,
    y=top_skills.index,
    ax=ax,
    palette="viridis",
)
ax.set_title("Top 15 Skills im deutschen Data-Jobmarkt", fontsize=13, fontweight="bold")
ax.set_xlabel("Anzahl Stellen mit Skill")
ax.set_ylabel("")
for i, v in enumerate(top_skills.values):
    ax.text(v + max(top_skills) * 0.005, i, str(v), va="center")
plt.tight_layout()
plt.show()

# Als Anteil ausdrücken
n_jobs = jobs["job_id"].nunique()
print(f"\n💡 Erkenntnis: SQL und Python bilden das Fundament — sie tauchen in jeweils")
print(f"   {top_skills.iloc[0]}/{n_jobs} ({top_skills.iloc[0]/n_jobs*100:.0f}%) bzw.")
print(f"   {top_skills.iloc[1]}/{n_jobs} ({top_skills.iloc[1]/n_jobs*100:.0f}%) der Stellen auf.")

## 2️⃣ Skill Co-Occurrence: Welche Tools kommen zusammen vor?

**Frage:** Wenn ein Job Python verlangt, was wird sonst noch gefragt? Eine Co-Occurrence-Matrix zeigt's.

**Methodik:** Für jeden Job die Skill-Liste sammeln, dann pro Skill-Paar zählen, wie oft sie zusammen auftauchen.

In [ ]:
from itertools import combinations
from collections import Counter

# Skill-Set pro Job aufbauen
skills_per_job = skills.groupby("job_id")["skill"].apply(set)

# Top 12 Skills für die Matrix (sonst zu unübersichtlich)
top12 = top_skills.head(12).index.tolist()

# Co-Occurrence-Counter
cooc = Counter()
for skill_set in skills_per_job:
    relevant = skill_set & set(top12)
    for a, b in combinations(sorted(relevant), 2):
        cooc[(a, b)] += 1

# Matrix bauen
matrix = pd.DataFrame(0, index=top12, columns=top12)
for (a, b), count in cooc.items():
    matrix.loc[a, b] = count
    matrix.loc[b, a] = count
for s in top12:
    matrix.loc[s, s] = top_skills[s]  # Diagonale: Eigenhäufigkeit

# Heatmap
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(matrix, annot=True, fmt="d", cmap="viridis", ax=ax, cbar_kws={"label": "Anzahl Jobs"})
ax.set_title("Skill Co-Occurrence Matrix (Top 12)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Welche Skill-Paare kommen am häufigsten zusammen?
top_pairs = sorted(cooc.items(), key=lambda x: -x[1])[:10]
print("Top 10 häufigste Skill-Kombinationen:")
for (a, b), count in top_pairs:
    print(f"  {a:<12s} + {b:<12s}  →  {count:>4} Jobs")

print("\n💡 Die klassische Data-Analyst-Kombination ist eindeutig erkennbar.")

## 3️⃣ Skills pro Rolle: Wer braucht was?

Eine **Skill-Heatmap pro Rollen-Cluster** macht sofort sichtbar, wie sich die Anforderungen unterscheiden.

In [ ]:
# Skills mit role_group joinen
skills_with_role = skills.merge(jobs[["job_id", "role_group"]], on="job_id")

# Auf Top-Rollen + Top-Skills beschränken
top_roles = jobs["role_group"].value_counts().head(6).index.tolist()
top_skills_list = top_skills.head(12).index.tolist()

# Pivot: Anteil pro Rolle
pivot = (
    skills_with_role[skills_with_role["role_group"].isin(top_roles)]
    .groupby(["skill", "role_group"])
    .size()
    .unstack(fill_value=0)
)
pivot = pivot.loc[pivot.index.isin(top_skills_list)]

# In Anteile umrechnen (% der Jobs in dieser Rolle, die diesen Skill nennen)
role_sizes = jobs.groupby("role_group")["job_id"].nunique()
pivot_pct = pivot.div(role_sizes, axis=1) * 100
pivot_pct = pivot_pct[top_roles]  # Reihenfolge

# Heatmap
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    pivot_pct, annot=True, fmt=".0f", cmap="viridis",
    cbar_kws={"label": "% der Jobs in der Rolle"}, ax=ax,
)
ax.set_title("Skill-Anforderungen pro Rolle (% der Jobs)", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Pro Rolle den dominantesten Skill ermitteln
print("Wichtigster Skill pro Rolle (höchster %-Anteil):\n")
for role in top_roles:
    if role in pivot_pct.columns:
        top = pivot_pct[role].idxmax()
        pct = pivot_pct[role].max()
        print(f"  {role:<25s}  →  {top:<12s} ({pct:.0f}%)")

## 4️⃣ Junior-spezifisches Skill-Profil

Was wird von **Berufseinsteigern** verlangt vs. dem Gesamtmarkt? Findet man als Junior eine andere Skill-Landschaft?

In [ ]:
# Skills aufteilen: Junior vs. nicht-Junior
junior_jobs = set(jobs[jobs["is_junior"] == 1]["job_id"])
n_junior = len(junior_jobs)
n_total = jobs["job_id"].nunique()

junior_skills = skills[skills["job_id"].isin(junior_jobs)]["skill"].value_counts()
all_skills = skills["skill"].value_counts()

comparison = pd.DataFrame({
    "junior_pct": (junior_skills / max(n_junior, 1) * 100).round(1),
    "total_pct": (all_skills / n_total * 100).round(1),
}).fillna(0)

comparison = comparison.loc[comparison["total_pct"] >= 5]  # nur Skills die häufig sind
comparison["diff"] = (comparison["junior_pct"] - comparison["total_pct"]).round(1)
comparison = comparison.sort_values("total_pct", ascending=False).head(12)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison))
ax.barh(x - 0.2, comparison["total_pct"], height=0.4, label="Gesamtmarkt", color="#94a3b8")
ax.barh(x + 0.2, comparison["junior_pct"], height=0.4, label="Junior", color="#10b981")
ax.set_yticks(x)
ax.set_yticklabels(comparison.index)
ax.set_xlabel("% der Jobs mit diesem Skill")
ax.set_title("Skill-Anforderungen: Junior vs. Gesamtmarkt", fontsize=13, fontweight="bold")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\n💡 Erkenntnis: Junior-Stellen verlangen meist die gleichen Top-Tools wie der")
print(f"   Gesamtmarkt — aber tendenziell weniger 'Spezial-Skills' wie Cloud-Stack.")

## 5️⃣ Remote-spezifisches Skill-Profil

Sind Remote-Stellen technologisch anders aufgestellt? Werden hier mehr Cloud/Modern-Stack-Skills verlangt?

In [ ]:
remote_jobs = set(jobs[jobs["is_remote_friendly"] == True]["job_id"])
n_remote = len(remote_jobs)

remote_skills = skills[skills["job_id"].isin(remote_jobs)]["skill"].value_counts()

remote_comp = pd.DataFrame({
    "remote_pct": (remote_skills / max(n_remote, 1) * 100).round(1),
    "total_pct": (all_skills / n_total * 100).round(1),
}).fillna(0)

remote_comp["über_/unter_repräsentiert"] = (remote_comp["remote_pct"] - remote_comp["total_pct"]).round(1)
remote_comp = remote_comp.loc[remote_comp["total_pct"] >= 3]
remote_comp = remote_comp.sort_values("über_/unter_repräsentiert", ascending=False)

print("Skills mit STÄRKERER Nachfrage in Remote-Stellen:")
print(remote_comp.head(8).to_string())
print()
print("Skills mit SCHWÄCHERER Nachfrage in Remote-Stellen:")
print(remote_comp.tail(5).to_string())

## ✅ Erkenntnisse

1. **SQL und Python sind das Fundament** — fast jede Data-Stelle nennt mindestens einen davon
2. **Klassische Kombi:** SQL + Python + Excel deckt 80%+ der Junior-Stellen ab
3. **Pro Rolle unterschiedliche Schwerpunkte:** Data Engineer → Cloud/AWS, Data Scientist → ML/Python, BI Analyst → Power BI/Tableau
4. **Junior-Stellen** verlangen größtenteils die gleichen Top-Tools wie der Gesamtmarkt — gute Nachricht für Einsteiger
5. **Remote-Stellen** sind technologisch nicht radikal anders, tendenziell aber moderner aufgestellt (mehr Cloud, Python, dbt)

→ Notebook **04** zieht alle Insights zu einer Markt-Geschichte zusammen, mit Fokus auf den Homeoffice-Markt.